# v2.11 — v2.04 + ±DI14 and the official SET50 index as inputs (validation only)

**On top of v2.04**, the adopted 3-class baseline (Siwat, 2026-11-10, `configs/v2_11.json`), 7 more input channels (18 → 25):
- **+DI14 / −DI14** from the ADX calculation (the same period as ADX).
- **SET50 index, 5 channels** built like the stock's own price channels: open / high / low / close as log returns against the previous index close, and the change of log(1 + volume). The same values for every stock on a date.
- **What stays:** everything else is exactly v2.04: price head 1 : 1, horizons 1 / 5 / 10, news-robust training (p 0.2), the same rows and seeds.

**Index data:** the official SET50 index from the frozen Siamchart archive (`set50_index_2019_2025.csv`, 1,698 days, the same trading days as the stock file). **Upload it to Drive `Senior/prices/` first.** For 2026 (post-hoc only), the prepare stage fetches Yahoo `^SET50.BK` and checks it against the frozen index on Oct–Dec 2025.

**Ablation:** the proposal's input list (scope 2, slide 9) does not include ±DI or index inputs.

**Runtime:** A100 or L4, Colab only (Linux). Runtime → Run all. About 1–1.5 h of training on an A100 (about 3.5 h on an L4), then about 15 min for the post-hoc step and the comparison.

**Adopt rule** (pre-registered, `claude/log-entry-2026-11-10-v2-11-design.md`), raw models vs v2.04, news on:
- **(a)** the paired 95% interval of v2.11 − v2.04 (3-horizon mean) is **not entirely below 0**;
- **(b)** at least **one horizon's** interval is **entirely above 0**.

Outputs: `Senior/results/v2xx/<GPU>/v2_11*`, `Senior/results/v2_11_posthoc/`, `Senior/results/v2_11_vs_v2_04.json`. Test block sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
from pathlib import Path
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
OUT = f'{RES}/v2xx/{GPU_TAG}'
V211 = f'{OUT}/v2_11.json'
POST = f'{RES}/v2_11_posthoc'                      # v2.11's post-hoc step
BASE = f'{RES}/v2_04_posthoc'                      # v2.04's post-hoc step, the comparison
BASE_RUN = f'{RES}/v2xx/A100/v2_04.json'           # v2.04's run (A100, 2026-10-27)
FWD_INPUTS = f'{RES}/forward_2026/inputs'
for f in [f'{BASE}/step0.json', f'{BASE}/trees.csv.gz', f'{BASE}/parts/val_blank.csv.gz', BASE_RUN, f'{FWD_INPUTS}/forward_prices.csv',
          f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv',
          f'{PRICE_DIR}/set50_index_2019_2025.csv']:
    assert os.path.exists(f), f'stop: missing {f} (upload set50_index_2019_2025.csv from the laptop to Drive Senior/prices/)' if 'set50_index' in f else f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)

def stream(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
for f in ['configs/v2_11.json', 'scripts/train_v2xx.py', 'src/train_utils.py', 'src/run_config.py', 'tests/test_v2_11.py', 'src/proposal_model.py', 'src/features_proposal.py', 'scripts/forward_test_v2xx.py',
          'scripts/step0_v2_03beta.py', 'scripts/compare_v2xx_posthoc.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.11 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
!cp "$PRICE_DIR/set50_index_2019_2025.csv" data/raw/prices/siamchart/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b',
            'data/raw/prices/siamchart/set50_index_2019_2025.csv': '2363de306a1e655b3ea3a2155883145c305f090303d7022e5b4cd3c202bc7139'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the frozen copy'
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
import run_config as RC
cfg = RC.load('configs/v2_11.json')
import features_proposal as FP
print('v2.11 channels:', len(FP.channels(cfg['inputs'])), FP.channels(cfg['inputs']))
print('loss:', json.dumps(cfg['loss']), '| training:', json.dumps(cfg['training']))

In [ ]:
# Tests (about 12 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_news_blank.py', 'tests/test_grad_diag.py', 'tests/test_no_price_head.py',
          'tests/test_step0_v2_03beta.py', 'tests/test_v2_11.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l and '    seed' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 1. Train v2.11 (v2.04 + ±DI14 + SET50 index channels): 3 seeds one after another
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
if done(V211):
    print('v2.11: already trained')
else:
    stream(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_11.json', '--out', V211])

In [ ]:
# 2. Summary beside v2.04 and the floors, and the learning curves
r = json.load(open(V211)); s, fl = r['summary']['model'], r['floors']
b = json.load(open(BASE_RUN))['summary']['model']
BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
print(f"{'':6}{'v2.11 F1':>18}{'v2.04':>8}{'bar':>7}{'continues':>11}{'reverses':>10}{'R2_OS':>10}{'v2.04 R2':>10}")
for h in ('h1', 'h5', 'h10'):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>11.4f} ± {m['macro_f1']['std']:.4f}{b[h]['macro_f1']['mean']:>8.4f}{BAR[h]:>7.3f}"
          f"{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}"
          f"{m['r2_os_vs_zero']['mean']:>+10.4f}{b[h]['r2_os_vs_zero']['mean']:>+10.4f}")
for x in r['runs']:
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'hold-out F1', x['best_inner_es_macro_f1'],
          'weights', x['params']['total'], 'minutes', round(x['seconds'] / 60, 1))
from IPython.display import Image, display
import importlib, plot_v2xx_training as PT
for png in (PT.plot(Path(V211)), PT.plot_calls(Path(V211))):
    if png:
        print('saved', png); display(Image(filename=str(png)))

In [ ]:
# 3. Post-hoc step for v2.11 (same script and settings as v2.03beta-v2.10; prepare also fetches the
#    2026 SET50 index and checks it): reproduction check, ts, blend (w on v2.11's
#    own inner hold-out), news blanked and placebo, on validation and 2026
common = ['--run-json', V211, '--fwd-inputs', FWD_INPUTS, '--out-dir', POST, '--label', 'v2.11 post-hoc']
for stage in ('prepare', 'predict', 'trees', 'analyze'):
    stream(['python', 'scripts/step0_v2_03beta.py', stage, *common])

In [ ]:
# 4. v2.11 vs v2.04 on the same rows, and the pre-registered adopt rule (mean not below; at least one horizon up)
stream(['python', 'scripts/compare_v2xx_posthoc.py', '--base', BASE, '--new', POST, '--rule', 'mean_held_any_horizon_up',
        '--out', f'{RES}/v2_11_vs_v2_04.json'])

**Reading the result**
- **ADOPT RULE** (last block): (a) the news-on 3-horizon mean is not significantly below v2.04, and (b) at least one horizon is significantly above it. Raw models.
- If the 2026 prepare stops on the SET50 check, send the message to Claude (a file can be given with `--set50-forward-csv`).
- Price R² is beside the trend scores (v2.04 has a price head).
- Send the printout to Claude.